# ArthoBodh: Unified BanglaBERT WSD Pipeline
### End-to-End Training on Combined Dataset (IndoWordNet 5,000 words + Raw Kaggle 100 words)

This notebook provides the unified training pipeline for **ArthoBodh**:
1. **Step 1: Dataset Generation & Harmonization** — Directly processes the raw Kaggle dataset (`data/raw/Bengali_WSD_Database`) into single-sentence contexts with `**target_word**` boundaries, and fetches 5,000 polysemous words from IndoWordNet (`pyiwn`, capped at 5 most important senses) to produce a unified 5,100-word catalog.
2. **Step 2: Load Combined Dataset** — Loads the consolidated 5,100-word dataset and sets up training/validation/test records.
3. **Step 3: Cross-Encoder Pair Construction** — Transforms each sentence and its candidate senses into `(context, gloss)` pairs.
4. **Step 4: BanglaBERT Cross-Encoder Model Architecture** — Loads pretrained BanglaBERT with a 1-logit regression scoring head.
5. **Step 5: Batch Scoring Matrix & Untrained Baseline** — Implements dynamic masked scoring over variable candidate senses and evaluates untrained accuracy.
6. **Step 6: Training & Validation** — Fine-tunes with AdamW, linear warmup, mixed precision, early stopping, and saves checkpoint to `checkpoints/banglabert-combined-v1`.
7. **Step 7: Dual Test Evaluation Breakdown** — Measures overall accuracy, IndoWordNet 5k accuracy, and Kaggle 100-word benchmark accuracy.
8. **Step 8: Interactive Disambiguation Demo** — Live prediction function (`predict_wsd`) and `ipywidgets` interface.

### 🚀 Google Colab Quickstart
If running in **Google Colab**:
1. **Enable GPU**: Go to **Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU**.
2. **Clone / Set Directory**:
```python
# Run this in Colab if you cloned ArthoBodh:
# !git clone https://github.com/Mehereen-1/ArthoBodh.git
# %cd ArthoBodh
```
3. **Mount Google Drive (Optional - to save checkpoints permanently)**:
```python
# from google.colab import drive
# drive.mount('/content/drive')
```

## Step 0: Environment Setup

Configures seed for reproducibility, detects hardware (CUDA GPU or CPU), and loads the Bengali text normalizer.

In [ ]:
import os
import sys
import re
import json
import time
import random
import unicodedata
import subprocess
from pathlib import Path

# Force UTF-8 on all platforms
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# ---- Automatic Directory Alignment for Google Colab ----
if Path("/content/ArthoBodh").exists() and Path.cwd() == Path("/content"):
    os.chdir("/content/ArthoBodh")
    print(f"Switched Colab working directory to: {Path.cwd()}")
elif Path("ArthoBodh").exists() and (Path("ArthoBodh") / "data").exists():
    os.chdir("ArthoBodh")
    print(f"Switched working directory to: {Path.cwd()}")
else:
    print(f"Current working directory: {Path.cwd()}")

# Colab Auto-Install Dependencies
try:
    import pyiwn
except ImportError:
    print("Installing pyiwn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyiwn"])
    import pyiwn

try:
    from normalizer import normalize as bn_normalize
except ImportError:
    try:
        print("Installing csebuetnlp normalizer...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/csebuetnlp/normalizer"])
        from normalizer import normalize as bn_normalize
    except Exception:
        bn_normalize = lambda s: unicodedata.normalize("NFC", str(s))

import torch
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__}")
print(f"Active Device: {device} | Mixed Precision (AMP): {use_amp}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Raw Kaggle Preprocessing & 5,000 IndoWordNet Extraction

This cell executes the end-to-end dataset creation from scratch:
1. **Raw Kaggle Dataset** (`data/raw/Bengali_WSD_Database`):
   - Reads `Sense1.txt` ... `Sense4.txt` across all 100 word directories.
   - Splits multi-sentence paragraphs into individual sentences containing the target word.
   - Annotates target boundaries with `**target_word**`.
2. **IndoWordNet 5,000 Words** (`pyiwn`):
   - Strictly excludes the 100 Kaggle benchmark words to guarantee zero data leakage.
   - Cleans definitions using synonym anchoring and boilerplate removal (`clean_gloss_normalized`).
   - **Sense Capping**: If any word has more than 5 senses, caps it to the **5 most important senses**.
   - Extracts verified example sentences containing `**target_word**`.
3. **Stratified Splitting & Export**:
   - 70% Train, 15% Validation, 15% Test saved to `data/processed_combined/dataset_splits.json`.

In [ ]:
# ---- Step 1: Full Dataset Preprocessing & Merger ----
import os
import sys
import re
import json
import random
import unicodedata
from pathlib import Path

_BENGALI_CHAR = r"ঀ-৿"
_ZERO_WIDTH = re.compile(r"[\u200B\u200C\u200D\uFEFF]")
_SPACES = re.compile(r"\s+")
_OBSCURE_TERMS = ['অসুর', 'বিরাটের পুত্র', 'বৈদীক যুগের', 'একটি কাব্যালঙ্কার']

def clean_spacing(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = _ZERO_WIDTH.sub("", text)
    return _SPACES.sub(" ", text).strip()

def normalize_text_clean(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\ufeff\u200b\u200c\u200d\r\t]", " ", text)
    text = text.replace("_", " ")
    text = re.sub(r'["`~^+=|\\/«»]', " ", text)
    text = text.replace("নদী বী ", "নদী বা ")
    text = text.replace("অবস্হিত", "অবস্থিত").replace("অবস্হা", "অবস্থা")
    text = text.replace("মুখথেকে", "মুখ থেকে").replace("ব্যাক্তি", "ব্যক্তি")
    text = re.sub(r"\s+", " ", text).strip(" _-—\t\r\n")
    return text

def clean_gloss_normalized(raw_gloss: str, lemmas: list, target_word: str) -> str:
    g = normalize_text_clean(raw_gloss)
    g = re.sub(r'^\s*\([^)]+\)\s*', '', g)
    if 'ফলস্বরূপ হওয়া' in g or 'শেষে তার' in g or 'ফলস্বরূপ হওয়া' in g:
        g = 'কাজের শেষ পরিণতি বা ফলাফল'
    elif 'ফুল থেকে উত্পন্ন হওয়া শাঁস' in g or 'ফুল থেকে উত্পন্ন হওয়া শাঁস' in g:
        g = 'গাছের রসালো খাদ্য বা বীজকোষ'
    elif 'পরিণাম রূপে প্রাপ্ত ফল' in g or g == 'পরিণাম রূপে প্রাপ্ত':
        g = 'কর্মের প্রতিফল বা বদলা'
    elif 'গণিতে কোনো সমস্যার' in g:
        g = 'গণিতের সমাধান বা প্রশ্নের উত্তর'
    else:
        prefixes = [
            r'^কোনো এমন বস্তু যা\s*', r'^এমন বস্তু যা\s*', r'^এমন বিষয় যা\s*', r'^এমন বিষয় যা\s*',
            r'^সেই প্রধান\s*', r'^মানুষের সেই সমূহ যাদের কাছে\s*', r'^সেই\s+', r'^কোনো\s+',
            r'^কোনও\s+', r'^একপ্রকার\s+', r'^একটি\s+', r'^একজন\s+', r'^এক\s+'
        ]
        for p in prefixes:
            g = re.sub(p, '', g, flags=re.IGNORECASE).strip()
        parts = re.split(r'\s+(?:যা|যার|যাকে|যাদের|যাতে|যেখানে|যখন|যে সময়|যে সময়|এবং যার)\s+', g)
        if parts[0] and len(parts[0].split()) >= 2:
            g = parts[0].strip()
        elif len(parts) > 1 and parts[1]:
            g = parts[1].strip()
    g = re.sub(r'\s+(?:বা|এবং|অথবা|ও|ইত্যাদি|প্রভৃতি|সেই)$', '', g).strip(' ,;:-—')
    words = g.split()
    if len(words) > 6:
        g = ' '.join(words[:6])
    norm_target = normalize_text_clean(target_word)
    cleaned_lemmas = [normalize_text_clean(l) for l in lemmas]
    other_lemmas = [l for l in cleaned_lemmas if l.lower() != norm_target.lower()]
    seen = set()
    uniq = [l for l in other_lemmas if not (l in seen or seen.add(l))]
    if uniq:
        syn_str = ', '.join(uniq[:2])
        if g:
            return g if g.startswith(syn_str) else f"{syn_str} ({g})"
        return syn_str
    return g

def extract_and_mark_sentence(paragraph: str, target: str) -> str:
    paragraph = clean_spacing(paragraph)
    target = clean_spacing(target)
    raw_sentences = [s.strip() for s in re.split(r'([।?!]+)', paragraph) if s.strip()]
    sentences = []
    i = 0
    while i < len(raw_sentences):
        s = raw_sentences[i]
        if i + 1 < len(raw_sentences) and re.match(r'^[।?!]+$', raw_sentences[i + 1]):
            s = s + " " + raw_sentences[i + 1]
            i += 2
        else:
            i += 1
        s = _SPACES.sub(" ", s).strip()
        if s:
            sentences.append(s)
    if not sentences:
        sentences = [paragraph]
    pattern = re.compile(rf"(?<![{_BENGALI_CHAR}])({re.escape(target)}[{_BENGALI_CHAR}]*)")
    matched_indices = [idx for idx, s in enumerate(sentences) if pattern.search(s)]
    if not matched_indices:
        matched_indices = [idx for idx, s in enumerate(sentences) if target in s]
    if not matched_indices:
        extracted = paragraph
    else:
        best_idx = matched_indices[0]
        extracted = sentences[best_idx]
        words = extracted.split()
        if len(words) < 6:
            if best_idx + 1 < len(sentences):
                extracted = extracted + " " + sentences[best_idx + 1]
            elif best_idx > 0:
                extracted = sentences[best_idx - 1] + " " + extracted
    cleaned = clean_spacing(extracted)
    if "**" not in cleaned:
        marked, count = pattern.subn(r'**\1**', cleaned, count=1)
        if count == 0:
            fallback = re.compile(rf"(\S*{re.escape(target)}\S*)")
            marked, count = fallback.subn(r'**\1**', cleaned, count=1)
        return marked if count > 0 else cleaned
    return cleaned

def mark_target_in_sentence(context: str, target: str, lemmas: list) -> tuple[str, bool]:
    norm_context = normalize_text_clean(context)
    norm_target = normalize_text_clean(target)
    pattern = rf'(?<![{_BENGALI_CHAR}]){re.escape(norm_target)}([{_BENGALI_CHAR}]*)(?![{_BENGALI_CHAR}])'
    match = re.search(pattern, norm_context)
    if match:
        return re.sub(pattern, rf'**{norm_target}\1**', norm_context, count=1), True
    for l in lemmas:
        l_clean = normalize_text_clean(l)
        if not l_clean:
            continue
        l_pat = rf'(?<![{_BENGALI_CHAR}]){re.escape(l_clean)}([{_BENGALI_CHAR}]*)(?![{_BENGALI_CHAR}])'
        m2 = re.search(l_pat, norm_context)
        if m2:
            return re.sub(l_pat, rf'**{l_clean}\1**', norm_context, count=1), True
    return norm_context, False

def parse_header(header_line: str):
    tags = re.findall(r'<([^>]+)>', header_line)
    word, sense_def = None, None
    for tag in tags:
        t = tag.strip()
        if t.startswith('word-') or t.startswith('word1-'):
            word = re.sub(r'^word\d*-', '', t).strip()
        elif re.match(r'sense\d+-', t):
            sense_def = re.sub(r'^sense\d+-', '', t).strip()
    return word, sense_def

def build_combined_dataset():
    # 1. Locate Raw Kaggle Directory
    raw_candidates = [
        Path("data/raw/Bengali_WSD_Database"),
        Path("../data/raw/Bengali_WSD_Database"),
        Path("/content/ArthoBodh/data/raw/Bengali_WSD_Database"),
        Path("ArthoBodh/data/raw/Bengali_WSD_Database"),
    ]
    raw_dir = next((p for p in raw_candidates if p.exists()), None)
    if raw_dir is None:
        raise FileNotFoundError(
            "Could not find 'data/raw/Bengali_WSD_Database'.\n"
            "If in Colab, run '%cd /content/ArthoBodh' in a cell first."
        )
    print(f"=== [1/3] Processing Raw Kaggle Dataset from {raw_dir} ===")
    folders = sorted([f for f in os.listdir(raw_dir) if (raw_dir / f).is_dir()],
                     key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 999)

    kaggle_catalog, kaggle_records, kaggle_words_set = {}, [], set()
    for folder in folders:
        folder_path = raw_dir / folder
        kaggle_catalog[folder] = {"target_word": None, "senses": {}, "source": "kaggle_raw"}
        for s_idx in [1, 2, 3, 4]:
            sfile = folder_path / f"Sense{s_idx}.txt"
            if not sfile.exists() or sfile.stat().st_size <= 3:
                continue
            with open(sfile, "r", encoding="utf-8-sig", errors="replace") as f:
                content = f.read().strip()
            if not content:
                continue
            paras = [p.strip() for p in re.split(r'\n\s*\n', content) if p.strip()]
            if not paras:
                continue
            first_lines = paras[0].splitlines()
            w, s_def = parse_header(first_lines[0])
            if kaggle_catalog[folder]["target_word"] is None and w:
                w_clean = clean_spacing(w)
                kaggle_catalog[folder]["target_word"] = w_clean
                kaggle_words_set.add(w_clean)
            kaggle_catalog[folder]["senses"][str(s_idx)] = clean_spacing(s_def or "")
            clean_paras = []
            if len(first_lines) > 1:
                body = '\n'.join(first_lines[1:]).strip()
                if body:
                    clean_paras.append(body)
            for p in paras[1:]:
                clean_paras.append(p)
            tw = kaggle_catalog[folder]["target_word"] or w
            for p_text in clean_paras:
                text_sentence = extract_and_mark_sentence(p_text, tw)
                if not text_sentence:
                    continue
                kaggle_records.append({
                    "folder": folder,
                    "target_word": tw,
                    "sense_num": s_idx,
                    "sense_label": s_idx - 1,
                    "sense_def": clean_spacing(s_def or ""),
                    "text": text_sentence,
                    "source_dataset": "kaggle_raw",
                })
    for r in kaggle_records:
        r["num_senses_for_word"] = len(kaggle_catalog[r["folder"]]["senses"])
    print(f"  Kaggle words: {len(kaggle_catalog)} | Clean sentence instances: {len(kaggle_records):,}")

    # 2. Fetch 5,000 IndoWordNet Words (Max 5 Senses)
    print("\n=== [2/3] Fetching 5,000 IndoWordNet Words (Max 5 Senses, Zero Leakage) ===")
    import pyiwn
    iwn = pyiwn.IndoWordNet(pyiwn.Language.BENGALI)
    MAX_IWN_WORDS = 5000
    MAX_SENSES_PER_WORD = 5
    iwn_catalog, iwn_records, excluded_count = {}, [], 0

    for w in iwn.all_words():
        if len(iwn_catalog) >= MAX_IWN_WORDS:
            break
        w_norm = normalize_text_clean(w)
        if w_norm in kaggle_words_set:
            excluded_count += 1
            continue
        try:
            w_syns = iwn.synsets(w)
        except Exception:
            continue
        if len(w_syns) < 2:
            continue
        filtered_syns = [s for s in w_syns if not any(t in s.gloss() for t in _OBSCURE_TERMS)]
        if len(filtered_syns) < 2:
            continue
        # Cap at 5 most important senses
        if len(filtered_syns) > MAX_SENSES_PER_WORD:
            filtered_syns = filtered_syns[:MAX_SENSES_PER_WORD]
        word_id = f"IWN_Word_{len(iwn_catalog) + 1}"
        senses_dict = {}
        for idx, s in enumerate(filtered_syns, 1):
            senses_dict[str(idx)] = clean_gloss_normalized(s.gloss(), s.lemma_names(), w_norm)
        word_records = []
        for idx, s in enumerate(filtered_syns, 1):
            for ex in s.examples():
                marked_ex, found = mark_target_in_sentence(ex, w_norm, s.lemma_names())
                if found:
                    word_records.append({
                        "folder": word_id,
                        "target_word": w_norm,
                        "sense_num": idx,
                        "sense_label": idx - 1,
                        "sense_def": senses_dict[str(idx)],
                        "text": marked_ex,
                        "source_dataset": "indowordnet",
                        "num_senses_for_word": len(filtered_syns),
                    })
        if len(set(r["sense_num"] for r in word_records)) >= 2:
            iwn_catalog[word_id] = {"target_word": w_norm, "senses": senses_dict, "source": "indowordnet"}
            iwn_records.extend(word_records)

    print(f"  IndoWordNet words: {len(iwn_catalog):,} | Verified sentences: {len(iwn_records):,}")
    print(f"  Excluded benchmark words: {excluded_count} | Max senses in any word: {max(len(v['senses']) for v in iwn_catalog.values())}")

    # 3. Combine and Split
    print("\n=== [3/3] Merging and Creating Stratified Splits (70% Train, 15% Val, 15% Test) ===")
    combined_catalog = {}
    combined_catalog.update(iwn_catalog)
    combined_catalog.update(kaggle_catalog)

    def make_stratified_split(records):
        shuffled = list(records)
        random.shuffle(shuffled)
        n = len(shuffled)
        n_train = int(0.70 * n)
        n_val = int(0.15 * n)
        return shuffled[:n_train], shuffled[n_train:n_train + n_val], shuffled[n_train + n_val:]

    iwn_tr, iwn_val, iwn_te = make_stratified_split(iwn_records)
    kag_tr, kag_val, kag_te = make_stratified_split(kaggle_records)
    train_combined = iwn_tr + kag_tr
    val_combined = iwn_val + kag_val
    test_combined = iwn_te + kag_te

    random.shuffle(train_combined)
    random.shuffle(val_combined)
    random.shuffle(test_combined)
    for r in train_combined: r["split"] = "train"
    for r in val_combined: r["split"] = "val"
    for r in test_combined: r["split"] = "test"

    combined_dataset = {
        "metadata": {
            "description": "Unified Bengali WSD Dataset combining 5,000 IndoWordNet words (max 5 senses) and 100 Raw Kaggle Benchmark words (sentence-extracted)",
            "total_words": len(combined_catalog),
            "total_instances": len(train_combined) + len(val_combined) + len(test_combined),
            "train_instances": len(train_combined),
            "val_instances": len(val_combined),
            "test_instances": len(test_combined),
            "words_indowordnet": len(iwn_catalog),
            "words_kaggle_raw": len(kaggle_catalog),
            "max_senses_per_word_indowordnet": MAX_SENSES_PER_WORD,
            "context_granularity": "Harmonized Sentence-Level",
            "target_marking_format": "**target_word**",
            "random_seed": SEED,
        },
        "catalog": combined_catalog,
        "train": train_combined,
        "val": val_combined,
        "test": test_combined,
    }

    out_candidates = [
        Path("data/processed_combined/dataset_splits.json"),
        Path("../data/processed_combined/dataset_splits.json"),
        Path("/content/ArthoBodh/data/processed_combined/dataset_splits.json"),
    ]
    output_path = out_candidates[0]
    for p in out_candidates:
        if p.parent.exists():
            output_path = p
            break
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(combined_dataset, f, ensure_ascii=False, indent=2)

    print(f"\nSUCCESS! Saved unified dataset to: {output_path}")
    print(f"  Total catalog words: {len(combined_catalog):,} (5,000 IndoWordNet + 100 Kaggle)")
    print(f"  Train sentences:     {len(train_combined):,} ({len(iwn_tr):,} IWN + {len(kag_tr):,} Kaggle)")
    print(f"  Val sentences:       {len(val_combined):,} ({len(iwn_val):,} IWN + {len(kag_val):,} Kaggle)")
    print(f"  Test sentences:      {len(test_combined):,} ({len(iwn_te):,} IWN + {len(kag_te):,} Kaggle)")
    print(f"  Total instances:     {len(train_combined) + len(val_combined) + len(test_combined):,}")

# Execute or load
dataset_check_candidates = [
    Path("data/processed_combined/dataset_splits.json"),
    Path("../data/processed_combined/dataset_splits.json"),
    Path("/content/ArthoBodh/data/processed_combined/dataset_splits.json"),
]
existing_dataset = next((p for p in dataset_check_candidates if p.exists()), None)
FORCE_REGENERATE = False

if existing_dataset and not FORCE_REGENERATE:
    print(f"Combined dataset already exists at: {existing_dataset.resolve()}")
    print("Skipping generation. (Set FORCE_REGENERATE = True if you want to rebuild from scratch).")
else:
    build_combined_dataset()


## Step 2: Load Combined Dataset

Loads the newly produced `data/processed_combined/dataset_splits.json` file into memory.

In [ ]:
dataset_path = Path("data/processed_combined/dataset_splits.json")
if not dataset_path.exists() and Path("../data/processed_combined/dataset_splits.json").exists():
    dataset_path = Path("../data/processed_combined/dataset_splits.json")

with open(dataset_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

catalog = dataset["catalog"]
train_records = dataset["train"]
val_records = dataset["val"]
test_records = dataset["test"]

print(f"Loaded Combined Dataset:")
print(f"  Catalog words:   {len(catalog):,}")
print(f"  Train instances: {len(train_records):,} ({sum(1 for r in train_records if r['source_dataset']=='indowordnet'):,} IWN + {sum(1 for r in train_records if r['source_dataset']=='kaggle_raw'):,} Kaggle)")
print(f"  Val instances:   {len(val_records):,} ({sum(1 for r in val_records if r['source_dataset']=='indowordnet'):,} IWN + {sum(1 for r in val_records if r['source_dataset']=='kaggle_raw'):,} Kaggle)")
print(f"  Test instances:  {len(test_records):,} ({sum(1 for r in test_records if r['source_dataset']=='indowordnet'):,} IWN + {sum(1 for r in test_records if r['source_dataset']=='kaggle_raw'):,} Kaggle)")
print(f"  Total instances: {len(train_records) + len(val_records) + len(test_records):,}")

## Step 3: Gloss Cross-Encoder Pair Construction

### Why Gloss Cross-Encoder?
In traditional classification, a model outputs a fixed set of classes (e.g. "Sense 1", "Sense 2"). That prevents the model from understanding words outside a tiny closed set.
In a **Gloss Cross-Encoder (GlossBERT)**:
- The candidate dictionary definitions (glosses) are supplied directly into the model's textual input.
- The model learns **semantic compatibility**: *Does definition $D$ match the usage of word $W$ in context $C$?*

---

### Understanding Input and Output Transformations

| Stage | Input | Transformation Logic | Output |
| :--- | :--- | :--- | :--- |
| **1. Context Preparation** (`prepare_context`) | Raw sentence with `**word**` markers | Replaces `**word**` with `" word "` quotation marks so attention heads focus on target. | Cleaned sentence with quoted target word. |
| **2. Gloss Preparation** (`prepare_gloss`) | `target_word`, `definition` | Combines target and meaning into standard gloss string: `f"{target_word} : {definition}"`. | Formatted definition string. |
| **3. Pair Construction** (`build_cross_encoder_pairs`) | Context, target, and dict of $K$ candidate senses | Forms $K$ pairs: `(context, target : definition_i)` for all $i \in \{1..K\}$. | `(first_segments, second_segments, sense_nums)` |
| **4. Tokenizer Encoding** (`encode_pairs`) | `firsts` (contexts), `seconds` (glosses) | BanglaBERT tokenizer applies `[CLS] context [SEP] target : definition [SEP]`. | PyTorch tensor dict (`input_ids`, `attention_mask`, `token_type_ids`). |

In [ ]:
BASE_MODEL = "csebuetnlp/banglabert"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

_MARK = re.compile(r"\*\*(.+?)\*\*")

def prepare_context(marked_text: str) -> str:
    """
    Converts '**word**' -> ' " word " ' (GlossBERT-style weak supervision marker).
    The quotes instruct BanglaBERT to attend closely to the target token.
    """
    text = _MARK.sub(lambda m: f' " {m.group(1).strip()} " ', marked_text)
    return re.sub(r"\s+", " ", bn_normalize(text)).strip()

def prepare_gloss(target_word: str, definition: str) -> str:
    """Formats candidate definition: '<target_word> : <definition>'."""
    return bn_normalize(f"{target_word} : {definition}")

def build_cross_encoder_pairs(context: str, target_word: str, senses: dict):
    """
    Constructs K text pairs for a word with K candidate senses:
      first segment:  Context with target word wrapped in quotes
      second segment: Target word and candidate sense definition
    """
    ctx = prepare_context(context)
    firsts, seconds, nums = [], [], []
    for num, definition in senses.items():
        firsts.append(ctx)
        seconds.append(prepare_gloss(target_word, definition))
        nums.append(int(num))
    return firsts, seconds, nums

def encode_pairs(firsts, seconds):
    """Tokenizes pairs with dynamic padding and truncation up to MAX_LEN."""
    return tokenizer(
        firsts,
        seconds,
        max_length=MAX_LEN,
        truncation="longest_first",
        padding=True,
        return_tensors="pt",
    )

# ---- Inspect sample pair formulation from both sources ----
sample_init = next(r for r in train_records if r["source_dataset"] == "kaggle_raw")
sample_iwn = next(r for r in train_records if r["source_dataset"] == "indowordnet")

print("=" * 80)
print("DEMO 1: Raw Kaggle Dataset Record Pair Construction")
print("=" * 80)
f_init, s_init, nums_init = build_cross_encoder_pairs(
    sample_init["text"], sample_init["target_word"], catalog[sample_init["folder"]]["senses"]
)
print(f"Target Word: '{sample_init['target_word']}' | True Gold Sense: {sample_init['sense_num']}")
for i in range(len(f_init)):
    print(f"  [Sense {nums_init[i]}] Context: {f_init[i]}")
    print(f"            Gloss:   {s_init[i]}")

enc_init = encode_pairs(f_init, s_init)
print(f"\nEncoded Tensor Shape: {enc_init['input_ids'].shape} (4 candidate sense pairs)")
print("First 30 token IDs:", enc_init["input_ids"][0][:30].tolist())

print("\n" + "=" * 80)
print("DEMO 2: IndoWordNet Record Pair Construction")
print("=" * 80)
f_iwn, s_iwn, nums_iwn = build_cross_encoder_pairs(
    sample_iwn["text"], sample_iwn["target_word"], catalog[sample_iwn["folder"]]["senses"]
)
print(f"Target Word: '{sample_iwn['target_word']}' | True Gold Sense: {sample_iwn['sense_num']}")
for i in range(len(f_iwn)):
    print(f"  [Sense {nums_iwn[i]}] Context: {f_iwn[i]}")
    print(f"            Gloss:   {s_iwn[i]}")

enc_iwn = encode_pairs(f_iwn, s_iwn)
print(f"\nEncoded Tensor Shape: {enc_iwn['input_ids'].shape} ({len(f_iwn)} candidate sense pairs)")

## Step 4: BanglaBERT Cross-Encoder Model Architecture

We load `csebuetnlp/banglabert` (ELECTRA-base pre-trained on large-scale Bengali corpus) with a single-output sequence classification head (`num_labels=1`).
- **Input**: Tokenized pair `[CLS] context [SEP] gloss [SEP]`.
- **Output**: A single scalar logit score representing how well that gloss fits the context.

In [ ]:
def load_model(path=BASE_MODEL):
    """Initializes BanglaBERT encoder with a single scalar classification head."""
    return AutoModelForSequenceClassification.from_pretrained(path, num_labels=1).to(device)

print(f"Loading base model from '{BASE_MODEL}' onto device '{device}'...")
model = load_model()
num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Architecture: {model.config.architectures[0] if model.config.architectures else 'AutoModel'}")
print(f"Total Parameters:     {num_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

## Step 5: Batch Scoring Matrix & Loss Computation

### Handling Variable Numbers of Senses
Different words have different numbers of senses (e.g. Word A has 2 senses, Word B has 4 senses).

1. **Joint Batch Flattening**: All candidate pairs for all sentences in the batch are encoded together in a single tensor pass.
2. **Matrix Reconstruction**: The scalar scores are placed into a `[batch_size, max_senses]` matrix using `(rows, cols)` coordinates.
3. **Masked Softmax (`-inf` Padding)**: Unused sense columns for words with fewer senses are padded with `-inf`:
   $$\text{logits}[i, j] = \begin{cases} \text{score}(\text{sentence}_i, \text{sense}_j) & \text{if sense exists} \\ -\infty & \text{if padded/invalid} \end{cases}$$
4. **Cross-Entropy**: Standard PyTorch `F.cross_entropy(logits, gold_labels)` evaluates loss only over the genuine candidate senses of that word.

In [ ]:
def score_batch(model, items):
    """
    Scores all candidate senses for a batch of sentences simultaneously.
    Returns a [batch_size, max_senses] tensor where unused sense slots are -inf.
    """
    firsts, seconds, rows, cols = [], [], [], []
    for row, (context, target, senses) in enumerate(items):
        f, s, nums = build_cross_encoder_pairs(context, target, senses)
        firsts += f
        seconds += s
        rows += [row] * len(nums)
        cols += list(range(len(nums)))

    enc = encode_pairs(firsts, seconds).to(device)
    pair_scores = model(**enc).logits.squeeze(-1).float()

    max_cols = max(cols) if cols else 0
    matrix = torch.full((len(items), max_cols + 1), float("-inf"), device=device)
    matrix[torch.tensor(rows, device=device), torch.tensor(cols, device=device)] = pair_scores
    return matrix

def to_items(batch):
    """Extracts (context, target_word, senses_dict) tuples for score_batch."""
    return [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in batch]

@torch.no_grad()
def evaluate(model, records, bs=16):
    """Computes loss and accuracy over a set of evaluation records."""
    model.eval()
    loss_sum, preds = 0.0, []
    for i in range(0, len(records), bs):
        batch = records[i : i + bs]
        labels = torch.tensor([r["sense_label"] for r in batch], device=device)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = score_batch(model, to_items(batch))
        loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
        preds += logits.argmax(-1).tolist()

    gold = [r["sense_label"] for r in records]
    acc = sum(p == g for p, g in zip(preds, gold)) / len(records)
    return loss_sum / len(records), acc, preds

# ---- Quick Untrained Baseline Verification ----
print("Evaluating untrained base model on sample validation records (baseline test)...")
sample_eval = val_records[:100]
untrained_loss, untrained_acc, _ = evaluate(model, sample_eval, bs=8)
print(f"Untrained Validation Loss:     {untrained_loss:.4f}")
print(f"Untrained Validation Accuracy: {untrained_acc * 100:.2f}%")
print("-> Untrained accuracy sits near random chance (~33-40%), confirming the model requires fine-tuning.")

## Step 6: Training (AdamW + Linear Warmup, FP16, Early Stopping on Val Accuracy)

- **Optimizer**: AdamW (`lr=2e-5`, `weight_decay=0.01` excluding bias & LayerNorm parameters).
- **Warmup Schedule**: Linear warmup over the first 10% of total training steps.
- **Mixed Precision (AMP)**: `torch.amp.GradScaler('cuda')` accelerates execution on CUDA GPUs.
- **Validation Checkpoint**: Evaluated against all 1,628 validation sentences after each epoch. Saves best checkpoint to `checkpoints/banglabert-combined-v1`.
- **Early Stopping**: Halts training if validation accuracy does not improve for `PATIENCE = 2` consecutive epochs.

In [ ]:
import collections

EPOCHS = 8
PATIENCE = 2          # stop after 2 epochs without val improvement
BATCH_SIZE = 8        # sentences per step (~20-30 candidate pairs)
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP = 0.10

no_decay = ("bias", "LayerNorm.weight")
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(param_groups, lr=LR)
steps_per_epoch = (len(train_records) + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# Resolve checkpoints directory relative to notebook or project root
checkpoint_dir = Path("checkpoints/banglabert-combined-v1")
if not checkpoint_dir.parent.exists() and Path("../checkpoints").exists():
    checkpoint_dir = Path("../checkpoints/banglabert-combined-v1")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"Combined Training Setup:")
print(f"  Train sentences: {len(train_records):,} | Val sentences: {len(val_records):,}")
print(f"  Batch size:      {BATCH_SIZE} | Steps per epoch: {steps_per_epoch:,}")
print(f"  Total steps:     {total_steps:,} | Warmup steps: {int(total_steps * WARMUP):,}")
print(f"  Checkpoint destination: {checkpoint_dir}\n")

best_val_acc, bad_epochs, history = 0.0, 0, []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    order = list(range(len(train_records)))
    random.shuffle(order)
    run_loss, correct, seen = 0.0, 0, 0

    for step, start in enumerate(range(0, len(order), BATCH_SIZE), 1):
        batch = [train_records[i] for i in order[start:start + BATCH_SIZE]]
        labels = torch.tensor([r["sense_label"] for r in batch], device=device)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = score_batch(model, to_items(batch))
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        run_loss += loss.item() * len(batch)
        correct += (logits.argmax(-1) == labels).sum().item()
        seen += len(batch)

        if step % 250 == 0 or step == steps_per_epoch:
            print(f"  epoch {epoch} step {step}/{steps_per_epoch} | running loss {run_loss/seen:.4f} | train acc {correct/seen*100:.1f}%")

    val_loss, val_acc, _ = evaluate(model, val_records, bs=BATCH_SIZE * 2)
    history.append({
        "epoch": epoch,
        "train_loss": run_loss / seen,
        "train_acc": correct / seen,
        "val_loss": val_loss,
        "val_acc": val_acc
    })
    print(f">>> Epoch {epoch} ({time.time()-t0:.0f}s) | train loss {run_loss/seen:.4f} acc {correct/seen*100:.2f}% "
          f"| val loss {val_loss:.4f} acc {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc, bad_epochs = val_acc, 0
        model.save_pretrained(checkpoint_dir)
        tokenizer.save_pretrained(checkpoint_dir)
        with open(checkpoint_dir / "training_info.json", "w", encoding="utf-8") as f:
            json.dump({
                "base_model": BASE_MODEL,
                "best_epoch": epoch,
                "val_accuracy": val_acc,
                "val_loss": val_loss,
                "total_words": len(catalog),
                "train_instances": len(train_records),
                "val_instances": len(val_records),
                "history": history
            }, f, ensure_ascii=False, indent=2)
        print(f"    -> Saved best checkpoint to {checkpoint_dir} (val acc {val_acc*100:.2f}%)\n")
    else:
        bad_epochs += 1
        print(f"    -> No validation improvement ({bad_epochs}/{PATIENCE})\n")
        if bad_epochs >= PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

## Step 7: Dual Test-Set Evaluation & Sub-Dataset Breakdown

Evaluates the best fine-tuned checkpoint across the full test set (1,975 sentences) and calculates:
1. **Overall Test Accuracy**: Generalization across all 1,975 held-out instances.
2. **IndoWordNet Test Split** (1,285 sentences): Evaluates accuracy on the 3,000 IndoWordNet vocabulary.
3. **Initial Benchmark Test Split** (690 sentences): Evaluates accuracy on the 100 baseline curated words.
4. **Accuracy Breakdown by Number of Senses** (2, 3, 4, 5+ candidate meanings).

In [ ]:
print(f"Loading best model checkpoint from '{checkpoint_dir}' for evaluation...")
best_model = load_model(checkpoint_dir)
test_loss, test_acc, test_preds = evaluate(best_model, test_records, bs=16)

# Sub-dataset subsets
iwn_test = [r for r in test_records if r["source_dataset"] == "indowordnet"]
init_test = [r for r in test_records if r["source_dataset"] == "kaggle_raw"]

_, iwn_acc, _ = evaluate(best_model, iwn_test, bs=16)
_, init_acc, _ = evaluate(best_model, init_test, bs=16)

print("=" * 70)
print("TEST EVALUATION RESULTS (banglabert-combined-v1)")
print("=" * 70)
print(f"Total Test Sentences:            {len(test_records):,}")
print(f"OVERALL COMBINED TEST ACCURACY:  {test_acc * 100:.2f}%  (loss: {test_loss:.4f})\n")
print(f"IndoWordNet Test Accuracy:       {iwn_acc * 100:.2f}%  ({len(iwn_test):,} sentences)")
print(f"Kaggle Benchmark Test Accuracy: {init_acc * 100:.2f}%  ({len(init_test):,} sentences)\n")

by_k = collections.defaultdict(lambda: [0, 0])
for r, p in zip(test_records, test_preds):
    k = len(catalog[r["folder"]]["senses"])
    k_str = str(k) if k <= 4 else "5+"
    by_k[k_str][0] += (p == r["sense_label"])
    by_k[k_str][1] += 1

print("Accuracy by Number of Candidate Senses:")
for k in sorted(by_k.keys(), key=lambda x: int(x.rstrip("+"))):
    c, n = by_k[k]
    print(f"  {k:>2} senses: {c/n*100:5.1f}%  ({c}/{n} correct)")

## Step 8: Live Disambiguation on New Sentences & Interactive Demo

Allows testing any arbitrary Bengali sentence on the fly using `predict_wsd()` or the interactive widget below.

In [ ]:
def mark_target(sentence: str, target: str) -> str:
    if "**" in sentence:
        return sentence
    pattern = rf'(?<![{_BENGALI_CHAR}]){re.escape(target)}([{_BENGALI_CHAR}]*)(?![{_BENGALI_CHAR}])'
    if re.search(pattern, sentence):
        return re.sub(pattern, rf'**{target}\1**', sentence, count=1)
    return re.sub(rf'(\S*{re.escape(target)}\S*)', r'**\1**', sentence, count=1)

def resolve_senses(target: str):
    t = bn_normalize(target)
    for v in catalog.values():
        if v["target_word"] == t:
            return v["senses"], v.get("source", "catalog")
    return None, None

@torch.no_grad()
def predict_wsd(sentence: str, target: str, show=True):
    senses, source = resolve_senses(target)
    if not senses:
        print(f"শব্দ '{target}' ৩,১০০ শব্দের ক্যাটালগে পাওয়া যায়নি।")
        return None

    best_model.eval()
    t0 = time.time()
    marked = mark_target(sentence, target)
    f, s, nums = build_cross_encoder_pairs(marked, target, senses)
    logits = best_model(**encode_pairs(f, s).to(device)).logits.squeeze(-1).float()
    probs = torch.softmax(logits, dim=-1).tolist()
    ranked = sorted(zip(nums, probs), key=lambda x: -x[1])

    if show:
        elapsed_ms = (time.time() - t0) * 1000
        print(f"টার্গেট শব্দ: '{target}' | উৎস: {source} | গতি: {elapsed_ms:.1f} ms")
        print(f"ইনপুট বাক্য:  {prepare_context(marked)}")
        for rank, (num, prob) in enumerate(ranked):
            marker = ">>>" if rank == 0 else "   "
            print(f"{marker} Sense {num:2d} ({prob * 100:5.1f}%): {senses[str(num)]}")
        print("-" * 75)
    return ranked

# ---- Quick Test on Diverse Examples ----
predict_wsd("আজ নদীর জল খুব ঠান্ডা", "জল")
predict_wsd("কঠোর পরিশ্রমের ফল সবসময় ভালোই হয়", "ফল")
predict_wsd("চাষি জমিতে মই দেওয়ার জন্য মই ও দড়ি নিয়ে এলো", "দড়ি")

# ---- Interactive Typebox Demo ----
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    target_input = widgets.Text(value="জল", description="শব্দ:", layout=widgets.Layout(width="40%"))
    sentence_input = widgets.Textarea(
        value="জল পড়ে পাতা নড়ে",
        description="বাক্য:",
        layout=widgets.Layout(width="90%", height="70px")
    )
    check_btn = widgets.Button(description="অর্থ নির্ণয় করুন", button_style="success", icon="search")
    out_box = widgets.Output()

    def on_check_clicked(_):
        with out_box:
            clear_output()
            w = target_input.value.strip()
            s = sentence_input.value.strip()
            if not w or not s:
                print("অনুগ্রহ করে বাক্য এবং টার্গেট শব্দ উভয়ই প্রদান করুন।")
                return
            predict_wsd(s, w)

    check_btn.on_click(on_check_clicked)
    display(widgets.VBox([
        widgets.HTML("<h3>ArthoBodh: Live Bengali WSD Disambiguation (Combined 3,100 Words)</h3>"),
        target_input, sentence_input, check_btn, out_box
    ]))
except ImportError:
    print("ipywidgets not installed. Run predict_wsd() directly in code cells.")

## Step 9: Save Checkpoint to Google Drive (Optional)
If running in Colab and you want to persist the trained checkpoint across sessions:
```python
# !mkdir -p /content/drive/MyDrive/ArthoBodh_Checkpoints
# !cp -r checkpoints/banglabert-combined-v1 /content/drive/MyDrive/ArthoBodh_Checkpoints/
# print("Checkpoint successfully backed up to Google Drive!")
```